# 04_04 – Проверка сопоставления C40 перед отбором

Согласовано сначала дополнительно проверить сопоставление. Правило заполненного C40 до первой попытки пока не применяем. Предварительный расчет сохранен отдельно; аналитические копии не созданы.

Проверяем весь файл на повторное использование одной отметки C40 у одного адреса и аэропорта. Разные посадки с одинаковым C40 – кандидаты на разбор, не доказанные дубли и не основание слияния. Дополнительно оцениваем неоднозначность контекста ICAO24/аэропорт/UTC-дата посадки среди строк с известным C40. Дата около полуночи не гарантирует корректность связи.

In [1]:
# Подключаем полный анализ в отдельном ядре
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
sys.path.insert(0,str(ROOT / 'europe/scripts'))
import audit_c40_matching as analysis

# Сохраняем группы и проекцию для ручной проверки без изменения меток и времен
c40_groups,c40_records,c40_contexts = analysis.main()
display(c40_groups)

{
  "input_sha256": "ddf11438bab44137242ffcf4da928418e116efd46defce58dc8a388b67942b6d",
  "input_unchanged": true,
  "rows_checked": 7984254,
  "rows_with_c40": 6331872,
  "rows_with_missing_context_keys": 0,
  "repeated_c40_groups": 147,
  "rows_sharing_c40": 294,
  "positive_rows_sharing_c40": 3,
  "groups_with_different_landing_times": 147,
  "groups_spanning_multiple_landing_dates": 0,
  "repeated_rows_c40_after_first": 3,
  "repeated_rows_c40_after_landing": 3,
  "same_day_multiple_landing_contexts_with_c40": 1229098,
  "rows_in_same_day_multiple_landing_contexts": 3171396,
  "rows_changed": 0,
  "rows_deleted": 0,
  "selection_applied": false,
  "trajectory_matching_verified": false,
  "scope": "Metadata audit; no independent trajectories or historical EU-NM records",
  "context_scope": "Same-day contexts counted only among rows with known C40"
}


,icao24,airport,C40_CROSS_TIME,rows,go_arounds,distinct_landing_times,landing_dates,first_landing,last_landing,c40_after_first_rows,c40_after_landing_rows,distinct_latitudes,distinct_longitudes,distinct_bearings,landing_span_minutes
0,04014e,EGLL,2019-01-09 19:21:42+00:00,2,0,2,1,2019-01-09 19:34:24+00:00,2019-01-09 19:46:40+00:00,0,0,1,1,1,12.266667
1,06a07b,LFPG,2019-01-30 04:49:36+00:00,2,0,2,1,2019-01-30 05:07:35+00:00,2019-01-30 05:24:27+00:00,0,0,1,1,1,16.866667
2,0a0026,EDDF,2019-02-01 11:17:25+00:00,2,0,2,1,2019-02-01 11:33:24+00:00,2019-02-01 11:53:05+00:00,0,0,1,1,1,19.683333
3,34134d,LEMD,2019-07-02 20:41:03+00:00,2,0,2,1,2019-07-02 20:56:18+00:00,2019-07-02 21:12:47+00:00,0,0,1,1,1,16.483333
4,341583,LEMD,2019-06-18 15:16:31+00:00,2,0,2,1,2019-06-18 15:36:02+00:00,2019-06-18 15:51:51+00:00,0,0,1,1,1,15.816667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,a51ede,EGSS,2019-07-24 01:03:37+00:00,2,0,2,1,2019-07-24 01:22:55+00:00,2019-07-24 01:39:29+00:00,0,0,1,1,1,16.566667
143,aac33e,LEMD,2019-07-25 13:06:44+00:00,2,0,2,1,2019-07-25 13:19:52+00:00,2019-07-25 13:34:27+00:00,0,0,1,1,1,14.583333
144,aacaac,LEMD,2019-03-09 08:55:19+00:00,2,0,2,1,2019-03-09 09:08:46+00:00,2019-03-09 09:28:02+00:00,0,0,1,1,1,19.266667
145,ad6b0e,LEMD,2019-09-14 13:38:20+00:00,2,0,2,1,2019-09-14 13:58:50+00:00,2019-09-14 14:18:06+00:00,0,0,1,1,1,19.266667


In [2]:
# Показываем временные связи повторно использованных отметок
with pd.option_context('display.max_rows',100,'display.max_columns',None):
    display(c40_records.sort_values(['icao24','airport','C40_CROSS_TIME','landing_time']).head(100))
display(c40_contexts.head(30))

,icao24,airport,landing_time,C40_CROSS_TIME,GoA,C40_CROSS_LAT,C40_CROSS_LON,C40_BEARING,first_attempt_time,source_row_position,c40_after_first_attempt,c40_after_landing,landing_utc_date
7,04014e,EGLL,2019-01-09 19:34:24+00:00,2019-01-09 19:21:42+00:00,False,51.090833,0.406944,125.109779,2019-01-09 19:34:24+00:00,52319,False,False,2019-01-09
9,04014e,EGLL,2019-01-09 19:46:40+00:00,2019-01-09 19:21:42+00:00,False,51.090833,0.406944,125.109779,2019-01-09 19:46:40+00:00,52383,False,False,2019-01-09
62,06a07b,LFPG,2019-01-30 05:07:35+00:00,2019-01-30 04:49:36+00:00,False,49.118611,3.550278,80.209015,2019-01-30 05:05:44+00:00,162501,False,False,2019-01-30
63,06a07b,LFPG,2019-01-30 05:24:27+00:00,2019-01-30 04:49:36+00:00,False,49.118611,3.550278,80.209015,2019-01-30 05:24:27+00:00,162555,False,False,2019-01-30
64,0a0026,EDDF,2019-02-01 11:33:24+00:00,2019-02-01 11:17:25+00:00,False,49.391389,8.843889,164.502009,2019-02-01 11:33:24+00:00,176921,False,False,2019-02-01
65,0a0026,EDDF,2019-02-01 11:53:05+00:00,2019-02-01 11:17:25+00:00,False,49.391389,8.843889,164.502009,2019-02-01 11:53:05+00:00,177049,False,False,2019-02-01
194,34134d,LEMD,2019-07-02 20:56:18+00:00,2019-07-02 20:41:03+00:00,False,40.028889,-4.207778,228.289885,2019-07-02 20:54:09+00:00,1223946,False,False,2019-07-02
197,34134d,LEMD,2019-07-02 21:12:47+00:00,2019-07-02 20:41:03+00:00,False,40.028889,-4.207778,228.289885,2019-07-02 21:12:47+00:00,1224008,False,False,2019-07-02
176,341583,LEMD,2019-06-18 15:36:02+00:00,2019-06-18 15:16:31+00:00,False,40.575278,-2.695000,80.819572,2019-06-18 15:34:23+00:00,1119681,False,False,2019-06-18
177,341583,LEMD,2019-06-18 15:51:51+00:00,2019-06-18 15:16:31+00:00,False,40.575278,-2.695000,80.819572,2019-06-18 15:51:51+00:00,1119773,False,False,2019-06-18


,icao24,airport,landing_utc_date,rows,distinct_c40_times,go_arounds
72,008ffa,EPWA,2019-06-26,2,2,1
1972,0100a3,LIMC,2021-08-07,2,2,0
3427,0100e4,LFPG,2021-11-06,2,2,0
3495,0100e4,LIMC,2023-04-29,2,2,0
3648,0100e5,LIMC,2019-01-13,2,2,0
3967,0100e6,LIMC,2019-03-24,2,2,0
4203,0100e7,LIMC,2019-03-03,2,2,0
6651,010140,EDDB,2019-04-06,2,2,0
7901,010147,LOWW,2019-06-06,2,2,0
9545,010193,LIMC,2019-03-17,2,2,0


## Что установлено и что остается неизвестным

Табличная проверка выявляет неоднозначные связи, но не удостоверяет истинный полет, первый/последний вход в ASMA или историческую геометрию. Для подтверждения нужна независимая траектория или исходная запись EUROCONTROL с правилами присоединения.

Нельзя исправлять C40 выбором ближайшей посадки только по этой таблице: это будет новая гипотеза, а не восстановленный факт. Временные флаги остаются диагностическими. До разбора не применяем правило отбора и сохраняем все полеты.

[Подробный разбор](../../reports/analytical_sample/04_04_review.md), [предыдущая проверка временных аномалий](../../reports/preparation/06_c40_review.md).